# Churn + CLV Prediction: Data Cleaning

**Data:** Online Retail II (UCI), a real UK online retailer, Dec 2009 - Dec 2011.
Output: two clean tables, `sales` (real purchases) and `returns` (real returns).

## 1. Load and combine the raw data

In [1]:
import pandas as pd

# Read ALL sheets of the Excel file (one sheet per year) into a dictionary
sheets = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name=None)

# Remember which sheet each row comes from, then stack the sheets into one table
for name, sheet in sheets.items():
    sheet["source_sheet"] = name
transactions = pd.concat(sheets.values(), ignore_index=True)

# Column names: lowercase, spaces become underscores ("Customer ID" -> "customer_id")
transactions.columns = transactions.columns.str.strip().str.lower().str.replace(" ", "_")

print(transactions.shape)

(1067371, 9)


## 2. Remove the overlap between the two sheets

In [2]:
# 1-9 Dec 2010 is in both sheets: keep it only from the newer sheet
keep = (transactions["source_sheet"] == "Year 2010-2011") | (transactions["invoicedate"] < "2010-12-01")
transactions = transactions[keep]

print(transactions.shape)

(1044848, 9)


## 3. Remove rows without a customer ID and duplicate rows

In [3]:
# A row without a customer ID cannot be followed over time
transactions = transactions.dropna(subset=["customer_id"])
transactions["customer_id"] = transactions["customer_id"].astype(int)   # 13085.0 becomes 13085

# Drop exact copies of a row (the first copy is kept)
transactions = transactions.drop_duplicates()

print(transactions.shape)

(797885, 9)


## 4. Remove non-product rows and separate sales from returns

In [4]:
# invoice and stockcode mix numbers and text (for example "C489449" and "79323P"): make them text
transactions = transactions.astype({"invoice": str, "stockcode": str})

# Remove rows that are not products (manual entries, postage, fees, adjustments, tests)
non_product = ["M", "POST", "C2", "DOT", "D", "BANK CHARGES", "ADJUST", "ADJUST2", "TEST001", "TEST002"]
transactions = transactions[~transactions["stockcode"].str.strip().str.upper().isin(non_product)]

# Cancelled invoices start with the letter C: they are the returns
is_cancelled = transactions["invoice"].str.startswith("C")
returns = transactions[is_cancelled]

# Real purchases: not cancelled, and a price above zero
sales = transactions[~is_cancelled & (transactions["price"] > 0)]

print("Sales:", sales.shape, "| Returns:", returns.shape)

Sales: (776596, 9) | Returns: (17603, 9)


## 5. Remove orders that were cancelled within one day

In [5]:
# An order cancelled within one day was never delivered, so it is not a real purchase.
# Example: invoice 581483 (80,995 items) was cancelled 12 minutes later by C581484.
# For every return line, find the latest earlier sales line of the same customer, product, quantity and price (at most 1 day before).
same_line = ["customer_id", "stockcode", "quantity", "price"]

cancelled = returns[same_line + ["invoicedate"]].rename(columns={"invoicedate": "return_date"})
cancelled["quantity"] = -cancelled["quantity"]    # returns have a negative quantity
cancelled["return_id"] = cancelled.index

bought = sales[same_line + ["invoicedate"]]
bought = bought.assign(sale_id=bought.index)

matched = pd.merge_asof(
    cancelled.sort_values("return_date"), bought.sort_values("invoicedate"),
    left_on="return_date", right_on="invoicedate", by=same_line,
    direction="backward", tolerance=pd.Timedelta(days=1),
)
matched = matched.dropna(subset=["sale_id"]).drop_duplicates(subset="sale_id")   # one return cancels one sales line

# Remove each matched sales line together with its cancellation line
sales = sales.drop(index=matched["sale_id"].astype(int))
returns = returns.drop(index=matched["return_id"])

print("Cancelled within a day:", len(matched), "| Sales:", sales.shape, "| Returns:", returns.shape)

Cancelled within a day: 1318 | Sales: (775278, 9) | Returns: (16285, 9)


## 6. Save the cleaned data

In [6]:
sales.to_parquet("../data/processed/sales.parquet", index=False)
returns.to_parquet("../data/processed/returns.parquet", index=False)
print("Saved")

Saved
